# Public cached-KD workflow

This notebook uses only code in this repository: streaming PTQ conversion, a content-bound teacher cache, and cached recovery with mandatory warm-model selection. No external release ZIP is needed.

The default is a **small CPU plumbing smoke** with random weights and synthetic token IDs. It verifies artifact creation and selection, not language quality. Set `SMOKE=False` for a real donor/corpus pilot; this downloads model weights and six public datasets unless local paths are supplied. Large-model memory fit and convergence require separate measured experiments. The public runner uses renormalized top-K teacher KD and an experimental homotopy schedule; it is not the archived external strict-v3 runner.

Run cells in order in a fresh Python environment. CUDA and CPU are supported by this workflow; CUDA requires a compatible PyTorch installation. Set `MN_SOURCE_REF` to a commit SHA to pin a later reproduction. During review its default is `fix/review-findings`. Existing local clones use their checked-out revision, recorded below.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, tempfile, time

SOURCE_REF = os.environ.get("MN_SOURCE_REF", "fix/review-findings")
HERE = Path.cwd()
LOCAL = HERE if (HERE / "pyproject.toml").exists() else HERE.parent
REPO = Path(os.environ.get("MN_REPO", str(LOCAL if (LOCAL / "pyproject.toml").exists() else HERE / "memory-native"))).resolve()
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout",
                    "https://github.com/kharkilirov1/memory-native.git", str(REPO)], check=True)
    subprocess.run(["git", "fetch", "origin", SOURCE_REF], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
# Existing clones keep their current revision. No automatic checkout or cleanup.
required = ["scripts/convert_streaming.py", "scripts/kd_teacher_cache.py",
            "scripts/kd_cached_recovery.py", "scripts/kd_cache_contract.py"]
if not all((REPO / name).is_file() for name in required):
    raise RuntimeError("This revision lacks the public cache contract; use fix/review-findings or its merged successor.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO) + "[dev,donor,corpus]"], check=True)
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Source revision:", REVISION)


Choose the experiment. All output goes into a new directory. `LOCAL_MODEL` must be a local safetensors snapshot; `LOCAL_DATA` must contain the donor-tokenized uint32 bins and `manifest.json` produced by `scripts/build_mix_corpus.py`. Do not substitute a different tokenizer's bins. Cache identities hash the actual donor and corpus contents; changed artifacts require rebuilding.

In [ ]:
SMOKE = True
MODEL_ID = "Qwen/Qwen2.5-0.5B"
LOCAL_MODEL = ""  # optional existing local snapshot for SMOKE=False
LOCAL_DATA = ""   # optional existing donor-tokenized mix for SMOKE=False

import importlib.metadata
import numpy as np
import torch
WORK = Path(tempfile.mkdtemp(prefix="memory-native-public-"))
DEVICE = "cpu" if SMOKE else ("cuda" if torch.cuda.is_available() else "cpu")
STEPS, SEQ, GROUP = (4, 16, 32) if SMOKE else (20, 128, 128)
BATCH, SEED, TOPK = 1, 0, 32

def run_script(name, **settings):
    env = dict(os.environ)
    env.update({key: str(value) for key, value in settings.items()})
    env["PYTHONPATH"] = str(REPO / "src")
    subprocess.run([sys.executable, str(REPO / "scripts" / name)], cwd=REPO, env=env, check=True)

if SMOKE:
    from transformers import Qwen2Config, Qwen2ForCausalLM
    torch.manual_seed(7)
    cfg = Qwen2Config(vocab_size=128, hidden_size=64, intermediate_size=128,
                     num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                     max_position_embeddings=128, tie_word_embeddings=False)
    MODEL_DIR, DATA_DIR = WORK / "donor", WORK / "data"
    Qwen2ForCausalLM(cfg).eval().save_pretrained(MODEL_DIR, safe_serialization=True)
    DATA_DIR.mkdir()
    rng = np.random.default_rng(7)
    for split, count in (("train", 4096), ("val", 256)):
        rng.integers(0, cfg.vocab_size, count, dtype=np.uint32).tofile(DATA_DIR / f"{split}_smoke.bin")
    (DATA_DIR / "manifest.json").write_text(json.dumps({
        "tokenizer": str(MODEL_DIR), "dtype": "uint32", "domains": {"smoke": {"share": 1.0}},
        "purpose": "synthetic plumbing only; no language-quality evidence"}), encoding="utf-8")
else:
    from huggingface_hub import snapshot_download
    MODEL_DIR = Path(LOCAL_MODEL).resolve() if LOCAL_MODEL else Path(snapshot_download(MODEL_ID))
    DATA_DIR = Path(LOCAL_DATA).resolve() if LOCAL_DATA else WORK / "data"
    if not LOCAL_DATA:
        env = dict(os.environ, MODEL=str(MODEL_DIR))
        subprocess.run([sys.executable, str(REPO / "scripts" / "build_mix_corpus.py"),
                        "--out", str(DATA_DIR), "--train-tokens", "240000", "--val-tokens", "2048"],
                       cwd=REPO, env=env, check=True)

BASE = dict(MODEL=MODEL_DIR, DATA_DIR=DATA_DIR, DEVICE=DEVICE, DTYPE="fp32",
            BATCH=BATCH, SEQ=SEQ, SEED=SEED, NUM_BLOCKS=0,
            SYNTHETIC_CALIBRATION=1 if SMOKE else 0)
(WORK / "run_manifest.json").write_text(json.dumps({
    "source_revision": REVISION, "source_dirty": bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)),
    "smoke": SMOKE, "model": str(MODEL_DIR), "data": str(DATA_DIR), "device": DEVICE,
    "torch": torch.__version__, "python": sys.version,
    "dependencies": {name: importlib.metadata.version(name) for name in
                     ("numpy", "transformers", "tokenizers", "safetensors", "accelerate", "huggingface-hub")},
    "steps": STEPS, "seq": SEQ,
    "batch": BATCH, "seed": SEED, "group": GROUP, "topk": TOPK,
}, indent=2), encoding="utf-8")
print("Artifacts:", WORK)


Convert the warm student. Resume checks include donor contents, calibration tokens and conversion options; each completed block has a hash. This smoke uses a short calibration set. A real quality experiment needs a fixed, representative calibration and validation protocol.

In [ ]:
STATE_DIR = WORK / "warm"
run_script("convert_streaming.py", **BASE, OUT_DIR=STATE_DIR, C=11, GROUP=GROUP,
           CALIB_BATCHES=2 if SMOKE else 16, MICRO_BATCH=1, RESUME=1,
           GRID="itf", SALIENT_FIRST=0.02, SALIENT_SCOPE="layer",
           IN_SWEEP_REFIT=1, CASCADE=1)


Cache the fixed teacher batches. The public cache stores top-K logits, not the strict-v3 tail-bucket/logsumexp format. Reusing this cache requires the same donor, corpus, sequence length, seed and batch order.

In [ ]:
CACHE = WORK / "teacher_cache"
run_script("kd_teacher_cache.py", **BASE, OUT=CACHE, STEPS=STEPS, TOPK=TOPK,
           SHARD_STEPS=2 if SMOKE else 10, CHUNK_ROWS=2 if SMOKE else 8)


Run recovery. Warm evaluation at alpha=0 is mandatory. A candidate is selected only when it improves the same validation aggregate by `MIN_IMPROVEMENT`; this runner does not implement the external strict-v3 per-domain acceptance gates. The configuration below is a small pilot, not a validated convergence recipe.

`best.pt` is a slim inference recovery artifact that must be overlaid on the recorded warm conversion and donor. It is not a standalone model or a full optimizer/RNG training-resume checkpoint.

In [ ]:
CKPT_DIR = WORK / "recovery"
run_script("kd_cached_recovery.py", **BASE, STATE_DIR=STATE_DIR, CACHE=CACHE,
           CKPT_DIR=CKPT_DIR, CKPT_TMP="", STEPS=STEPS, STATS_SCOPE="group", DECIMATION=1,
           COUNTER_LR_START=1.25e-4, COUNTER_LR_END=1e-5,
           SCALE_LR_START=2.5e-5, SCALE_LR_END=5e-6,
           FP_LR=1e-5, GRAD_CLIP=0.5, KD_T=1.0, CE_ALPHA=1.0,
           HOMOTOPY_HOLD=0.2, HOMOTOPY_END=0.9,
           GRAD_CKPT=0 if SMOKE else 1, FREEZE_EMBED=1, SPLIT_GPUS=0,
           EVAL_EVERY=2 if SMOKE else 10, EVAL_MAX_TOKENS=256 if SMOKE else 2048,
           EVAL_AT_START=1, MIN_IMPROVEMENT=0.001, LOG_EVERY=1, LOGSUMEXP_CHUNK=8192)


Read the decision rather than assuming that the last training step won. `USE_WARM_STATE.txt` means the original warm artifact remains selected. Keep the donor, warm state, cache manifests, metrics and selection metadata together for review. Random smoke perplexities carry no language-quality claim.

In [ ]:
selection = json.loads((CKPT_DIR / "selection.json").read_text())
selected_artifact = json.loads((CKPT_DIR / "selected_artifact.json").read_text())
print(json.dumps(selection, indent=2))
print(json.dumps(selected_artifact, indent=2))
print("All artifacts:", WORK)

# Reload the selected artifact and check that its validation metric matches.
sys.path.insert(0, str(REPO / "scripts"))
from kd_cached_recovery import restore_selected_artifact
from recovery_session import DomainMix
from memory_native.eval import perplexity
from memory_native.recovery.runtime import metric_from_ppl
import math
selected_model = restore_selected_artifact(CKPT_DIR / "selected_artifact.json", device=DEVICE)
validation = DomainMix(DATA_DIR, seq=SEQ, batch=BATCH, seed=SEED).val_batches(
    DEVICE, max_tokens=256 if SMOKE else 2048)
reloaded_ppl = {f"ppl_{name}": perplexity(selected_model, batches)
                for name, batches in validation.items()}
reloaded_metric = metric_from_ppl(reloaded_ppl)
assert math.isclose(reloaded_metric, selection["selected_metric"], rel_tol=1e-5, abs_tol=1e-5)
print("Reloaded selected metric:", reloaded_metric)
